# 2-semantic-analysis-openai.ipynb

This notebook uses the OpenAI API to perform a semantic study on a dataset of news articles collected from Google Alerts.

We focus on rows from a Google Spreadsheet, specifically columns:
- G: Detected Language
- H: Detected Country
- I: Extracted Text

For each valid entry, the notebook uses OpenAI to:
1. Summarize the text.
2. Extract a "semantic universe" — a list of themes, concepts, and related areas.
3. Generate a list of relevant keywords.

This step builds on previously cleaned and language-processed data (see `0-clean-duplicates.ipynb` and `1-url-language-country-analyzer.ipynb`).

# Step 1: Install & import dependencies

In [1]:
!pip install --upgrade openai google-api-python-client google-auth google-auth-oauthlib python-dotenv pandas

In [2]:
import os
import pandas as pd
import logging
from dotenv import load_dotenv
from googleapiclient.discovery import build
from google.oauth2 import service_account
from openai import OpenAI

# Step 2: Load credentials and connect to Google Sheets

In [5]:
# Load environment variables
load_dotenv()

SERVICE_ACCOUNT_FILE = os.getenv("SERVICE_ACCOUNT_FILE")
SPREADSHEET_ID = os.getenv("SPREADSHEET_ID")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) # You must add this to your .env file manually

# Define scope and authenticate with Google
SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]
creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
sheet = build("sheets", "v4", credentials=creds).spreadsheets()

# Step 3: Load all rows from Sheet0 and filter by `wordAlert == "bioart"`

To avoid analyzing unrelated data, we only process rows where the `wordAlert` column contains `"bioart"`.  
This preserves alignment with the main dataset and allows us to update only the correct rows later.

In [6]:
# Read the full sheet (A to I to cover wordAlert to text)
result = sheet.values().get(
    spreadsheetId=SPREADSHEET_ID,
    range="Sheet0!A2:I"
).execute()

rows = result.get("values", [])

# Define column names (adjust as needed)
columns = [
    "wordAlert",  # A
    "link",       # B
    "date",       # C
    "source",     # D
    "title",      # E
    "description",# F
    "detected-language",  # G
    "detected-country",   # H
    "text"                # I
]

# Normalize rows to fixed length
normalized = [row + [""] * (len(columns) - len(row)) for row in rows]
df_all = pd.DataFrame(normalized, columns=columns)

# Filter only rows where wordAlert == "bioart"
bioart_mask = df_all["wordAlert"].str.strip().str.lower() == "bioart"
df_bioart = df_all[bioart_mask].copy()

# Store the original row indexes (to write back later)
bioart_indices_in_sheet = df_bioart.index.tolist()

# Keep only columns needed for semantic analysis
df_bioart = df_bioart[["detected-language", "detected-country", "text"]]
df_bioart.head()

,detected-language,detected-country,text
35,hr,Unknown,Bio Awaking: Spoj umetnosti i nauke za održivu...
36,de,Unknown,Für Seehamer Röster ist Kaffee eine Lebenseins...
224,hr,Unknown,Šta se dešava kada umetnici uđu u naučne labor...
225,el,Unknown,Εγκαίνια της έκθεσης “Το Μεταλλαξιογόνο Μέλλον...
226,pt,Unknown,Editora Roncarati - ALERTAS ANVISA EM 24.05.20...


# Step 4: Analyze each row, define prompt and semantic analysis logic

We use the OpenAI API to:
- Complete missing `language` and `country` if marked as "unknown"
- Generate a short summary
- Suggest a semantic universe
- List relevant keywords

This step returns 5 new values per row, matching the columns:  
`language`, `country`, `summary`, `semantics`, and `keywords`


In [7]:
def analyze_with_openai_flexible(detected_lang, detected_country, text):
    if not text.strip():
        return {
            "language": "No Text",
            "country": "No Text",
            "summary": "No Summary",
            "semantics": "No Semantics",
            "keywords": "No Keywords"
        }

    MAX_TOKENS_TEXT = 10000
    text = text[:MAX_TOKENS_TEXT]

    lang_map = {
        "en": "English", "pt": "Portuguese", "de": "German",
        "fr": "French", "es": "Spanish", "it": "Italian",
        "zh": "Chinese", "ja": "Japanese"
    }

    lang_final = lang_map.get(detected_lang.lower().strip(), "") if detected_lang else ""
    ask_language = not lang_final

    country_final = detected_country.strip() if detected_country and detected_country.lower() != "unknown" else ""
    ask_country = not country_final

    prompt_parts = []

    if ask_language:
        prompt_parts.append("1. Detect the language of the text.")
    else:
        prompt_parts.append(f"1. Use language: {lang_final}")

    if ask_country:
        prompt_parts.append("2. Detect the country of origin or main reference in the text.")
    else:
        prompt_parts.append(f"2. Use country: {country_final}")

    prompt_parts.extend([
        "3. Write a 1–2 sentence summary.",
        "4. List the semantic universe (thematic fields or domains).",
        "5. Provide a list of 5–10 keywords.",
        "Return all of this in a JSON object with keys: language, country, summary, semantics, keywords."
    ])

    prompt = f"""You are a semantic analysis engine.

Text:
{text}

{" ".join(prompt_parts)}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            # model="gpt-4-1106-preview",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=600
        )

        import json
        raw = response.choices[0].message.content
        parsed = json.loads(raw)

        return {
            "language": parsed.get("language", lang_final or "Unknown"),
            "country": parsed.get("country", country_final or "Unknown"),
            "summary": parsed.get("summary", "Missing"),
            "semantics": parsed.get("semantics", "Missing"),
            "keywords": parsed.get("keywords", "Missing"),
        }

    except Exception as e:
        logging.warning(f"OpenAI failed: {e}")
        return {
            "language": lang_final or "Error",
            "country": country_final or "Error",
            "summary": "Error",
            "semantics": "Error",
            "keywords": "Error"
        }


# Step 5: Apply to dataset

In [9]:
# Apply row by row, passing all required arguments
results = df_bioart.apply(
    lambda row: analyze_with_openai_flexible(row["detected-language"], row["detected-country"], row["text"]),
    axis=1
)

# Convert the list of dictionaries into a DataFrame
results_df = pd.DataFrame(results.tolist(), index=df_bioart.index)

# Join results with original dataframe
df_bioart = pd.concat([df_bioart, results_df], axis=1)

df_bioart.head()


KeyboardInterrupt: 

# Step 6: Write results back to Google Sheets (e.g., columns J, K, L)

In [ ]:
# Prepare data to write (summary, semantic universe, keywords)
values_to_write = df[["summary", "semantic_universe", "keywords"]].values.tolist()

# Write to columns L, M, N (i.e., columns 12–14) starting from row 2
sheet.values().update(
    spreadsheetId=SPREADSHEET_ID,
    range=f"Sheet0!L2:N{len(values_to_write)+1}",
    valueInputOption="RAW",
    body={"values": values_to_write}
).execute()

print("✅ Semantic results written to columns L–N in Sheet0.")
